<a href="https://colab.research.google.com/github/alvarosamp/TCC/blob/main/ObsPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Instalando a biblioteca
!pip install obspy


Informações sobre os eixos :

Z = Componente vertical - BHZ
Eixo vertical (baixo e cima)

O que mede ?
- Movimento do solo para cima e para baixo
- Muito sensivel a ondas P ( ondas primárias)
- Geralmente é a componente mais 'forte' em eventos distantes

Componente Norte-sul - BHN
N = Eixo horizonal norte sul
O que mede?
- Movimento horizontal na direção nrote <-> sul
- Importante para calcular dimensão da fonte
- Muito inlfuenciados por onda S

Componente Leste-Oeste BHE:
E = eixo horizonatal Leste <-> Oeste
O que mede?
- Moviemnto horionaltal perpendicular ao Norte e Sul
- Complementa a BHN
- Junto com BHN permite reconstruir o vetor horizontal

Aplicando no contexto do tcc:
VOce provavelmente usa :
PREFERRED_CHANNELS = {"BHZ", "BHN", "BHE"}

Modelos sismicos usam as 3 componetnes
Redes neurais podem usar como input 3 canais (tipo RGB)
Da para calcular a magnitude vetorial :
Magnitude = sqrt(z²+n²+e²)





In [2]:
#Carregando as bibliotecas
import obspy
from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client
from obspy import Trace, Stream
import matplotlib.pyplot as plt
import numpy as np

#Configurando os graficos
plt.style.use('ggplot')
%matplotlib inline

In [3]:
#Criando dados sismicos simulados
#Vamos criar um terremoto para entender
print('Criando a estacao')
#Parametros na nossa estação
taxa_amostragem = 20.0 #20 amostras por segundo
duracao = 30 #Segundos
n_amostras = int(taxa_amostragem * duracao) #Quantidade de amostras

#Vetor de tempo ( 0 a 30 segundos)
tempo = np.arange(n_amostras) / taxa_amostragem

"""
Qual é a diferença de duração e tempo ?
No caso, definimos duração como 30, logo, temos 30 segundos de evento sismico.
Sendo um parametro físico de experimento/simulação

O que é o tempo?
  Nesse exemplo estamos criando o vetor de tempo real, ponto a ponto.
  Ou sjea, Em qual instante cada amostra foi medida

De forma matemática
  taxa_amostragem = 20.0 #20Hz
  duracao = 30 # Segundos
  Logo,: n_amostras = 600, ou seja, teremos 600 pontos

  Agora:
  np.arange(n_amostras)
  gera: [0,1,2,3,4,5,...,599]
  dividindo por 20, temos : [0/20, 1/20],
  ou seja, começa em 0 segundos, termina em aprox 30, passo de 0.05]


  Resumo: Duracao -> Tempo total do sinal -> Escalar
          Tempo -> Vetor com instante de cada amostra -> Vetor

  No meu projeto real:
        Duração = taanho da janela ( ex: 60s no codigo)
        Tempo = Eixo x do gráfico
        taxa amostragem = frequencia do sensor

        Sempre precisamos manter os 3 conectados
        n_amostras = taxa_amostragem * duracao
"""
# Componente Vertical (Z) - onda principal
print("\n📊 Gerando componente vertical (BHZ)...")
sinal_z = (3.0 * np.sin(2 * np.pi * 0.5 * tempo) +   # Onda lenta (0.5 Hz)
           2.0 * np.sin(2 * np.pi * 2.0 * tempo) +   # Onda média (2 Hz)
           1.0 * np.sin(2 * np.pi * 5.0 * tempo) +   # Onda rápida (5 Hz)
           0.8 * np.random.randn(n_amostras))        # Ruído

# Componente Norte-Sul (N)
print("📊 Gerando componente norte-sul (BHN)...")
sinal_n = (2.5 * np.sin(2 * np.pi * 0.5 * tempo + 0.7) +
           1.5 * np.sin(2 * np.pi * 2.0 * tempo - 0.3) +
           0.5 * np.sin(2 * np.pi * 5.0 * tempo + 0.9) +
           0.8 * np.random.randn(n_amostras))

# Componente Leste-Oeste (E)
print("📊 Gerando componente leste-oeste (BHE)...")
sinal_e = (2.2 * np.sin(2 * np.pi * 0.5 * tempo - 0.5) +
           1.8 * np.sin(2 * np.pi * 2.0 * tempo + 0.4) +
           0.7 * np.sin(2 * np.pi * 5.0 * tempo - 0.8) +
           0.8 * np.random.randn(n_amostras))

print(f"\n✅ Dados criados com sucesso!")
print(f"   {n_amostras} amostras por componente")
print(f"   Taxa: {taxa_amostragem} Hz")
print(f"   Duração: {duracao} segundos")

Criando a estacao

📊 Gerando componente vertical (BHZ)...
📊 Gerando componente norte-sul (BHN)...
📊 Gerando componente leste-oeste (BHE)...

✅ Dados criados com sucesso!
   600 amostras por componente
   Taxa: 20.0 Hz
   Duração: 30 segundos


In [4]:
#Entendendo o trace
#trace = um único canal contino
"""
Trace = traço = Um UNICO canal continuo de dados sismicos
- Representa uma componente vertical (ex : vertical, norte ou leste)
- Tem dados igualmente espaços no tempo
- Contém metadados (informaçoes sobre a gravaçao)

"""

#Criando primeiro trace (componente vertical)
trace_z = Trace(data = sinal_z)
#Preenhendo metadados (informaçoes importantes )
trace_z.stats.station = 'Meus' #Nome da estaçao
trace_z.stats.network = 'XX' #Codigo da rede
trace_z.stats.channel = 'BHZ' # B=banda larga, H=alta ganho, Z=vertical
trace_z.stats.sampling_rate = taxa_amostragem
trace_z.stats.starttime = UTCDateTime("2024-01-15T12:00:00")
